# 🏙️ محاكي أسلوب أمين منطقة الرياض — النسخة النهائية

> يحوّل أي نص ليخرج بأسلوب كتابة أمين منطقة الرياض — معتمداً على بيانات حقيقية + RAG ذكي بالـ Embeddings + بصمة أسلوبية مفصّلة + مراجعة تلقائية.

---

## 🗺️ خريطة الـ Notebook

| القسم | المحتوى |
|-------|---------|
| **١** | التثبيت والإعداد |
| **٢** | تحضير البيانات |
| **٣** | نظام RAG الذكي (Embeddings + Bonus) |
| **٤** | البصمة الأسلوبية الكاملة |
| **٥** | التحويل — الدوال الرئيسية |
| **٦** | الواجهة التفاعلية |
| **٧** | إضافة بيانات جديدة |

### ترتيب التشغيل الصحيح
```
١. شغّل القسم ١ (التثبيت) — مرة واحدة فقط
٢. شغّل القسم ٢ (تحميل البيانات)
٣. شغّل القسم ٣ (بناء RAG) — مرة واحدة فقط
٤. شغّل القسم ٤ (البصمة الأسلوبية)
٥. شغّل القسم ٥ (الدوال)
٦. شغّل القسم ٦ (الواجهة) واستخدمها
٧. القسم ٧ اختياري — لإضافة بيانات جديدة
```

---
## ١. التثبيت والإعداد

In [ ]:
# تثبيت المكتبات المطلوبة — شغّل مرة واحدة فقط
import subprocess, sys

pkgs = ["groq", "sentence-transformers", "scikit-learn", "pandas", "numpy", "ipywidgets"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)

print("✅ جميع المكتبات جاهزة")

In [ ]:
import pandas as pd
import numpy as np
import json, re, time
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# ── مفاتيح API ──────────────────────────────────────────────────
GROQ_API_KEY = __import__("os").environ.get("GROQ_API_KEY", "")
client       = Groq(api_key=GROQ_API_KEY)
MODEL        = "llama-3.3-70b-versatile"   # للكتابة الإبداعية
EVAL_MODEL   = "llama-3.1-8b-instant"      # للتحليل السريع
CSV_PATH     = "riyadh_data_enriched.csv"  # ← تأكد أن الملف موجود

print("✅ الإعداد اكتمل")

In [ ]:
# تحميل نموذج Embeddings — يحمّل مرة واحدة فقط (~560MB أول مرة)
print("⏳ تحميل نموذج Embeddings...")
EMBED_MODEL = SentenceTransformer("intfloat/multilingual-e5-base")
print("✅ النموذج جاهز")

---
## ٢. تحضير البيانات

In [ ]:
df = pd.read_csv(CSV_PATH)
df['clean_text'] = df['clean_text'].fillna('')
print(f"✅ تم تحميل {len(df)} نص")

cols_to_show = [c for c in ["id","title_clean","type_ar","topic","opening_pattern","has_stats","has_religious"] if c in df.columns]
display(df[cols_to_show].head(10))

In [ ]:
# إحصائيات سريعة عن الداتا
if 'topic' in df.columns:
    print("── توزيع المواضيع ──────────────────")
    print(df['topic'].value_counts().to_string())
if 'opening_pattern' in df.columns:
    print("\n── توزيع أنماط البداية ─────────────")
    print(df['opening_pattern'].value_counts().sort_index().to_string())
if 'has_stats' in df.columns:
    print(f"\nنصوص بإحصائيات    : {df['has_stats'].sum()}")
if 'has_religious' in df.columns:
    print(f"نصوص بعبارات دينية: {df['has_religious'].sum()}")

---
## ٣. نظام RAG الذكي

> يجمع **Embeddings دلالية عميقة** + **bonus للموضوع** + **bonus للخصائص** للحصول على أدق الأمثلة

In [ ]:
class SmartRAG:
    """
    نظام استرجاع ذكي يعتمد على:
    - Embeddings دلالية (multilingual-e5-base) للتشابه العميق
    - تصنيف تلقائي للموضوع والخصائص بالـ AI
    - نظام Bonus لتعزيز الأمثلة ذات الصلة
    """

    def __init__(self, dataframe):
        self.df = dataframe.copy()
        self._build_embeddings()

    def _build_embeddings(self):
        print("⏳ بناء Embeddings...")
        texts = [
            "passage: " + str(row.get("title_clean", "")) + " " + str(row.get("clean_text", ""))[:400]
            for _, row in self.df.iterrows()
        ]
        self.embeddings = EMBED_MODEL.encode(
            texts, show_progress_bar=True, normalize_embeddings=True
        )
        print(f"✅ تم بناء {len(self.embeddings)} embedding")

    def _detect_query_meta(self, query):
        """يصنّف النص المدخل: الموضوع + الخصائص + نمط البداية المناسب"""
        prompt = f"""المهمة: صنّف النص وأرجع JSON فقط.

النص: {query[:400]}

اختر topic من القائمة:
البنية التحتية     ← طرق، جسور، أنفاق، قطار، مواصلات
جودة الحياة        ← حدائق، ترفيه، خدمات سكنية، نظافة، أحياء
التكريم والجوائز   ← جوائز، تكريم موظفين، شهادات تقدير
الزيارات والفعاليات← زيارات رسمية، مؤتمرات، معارض، منتديات
رؤية 2030          ← برامج رؤية، تحول وطني، مستهدفات المملكة
البيئة والاستدامة  ← مبادرات خضراء، تشجير، مياه، تغير مناخي
التقنية والابتكار  ← تطبيقات، ذكاء اصطناعي، مدن ذكية، رقمنة
الإدارة البلدية    ← قرارات إدارية، اجتماعات، لجان، سياسات
المجتمع والثقافة   ← تراث، هوية، مناسبات وطنية
الإنجازات والأرقام ← تقارير إنجاز، أرقام إحصائية، نتائج

- has_stats: true إذا فيه أرقام أو إحصائيات
- has_religious: true إذا فيه آيات أو أحاديث
- desired_opening: رقم 1-7 (الأنسب للموضوع)

مثال: {{"topic": "جودة الحياة", "has_stats": true, "has_religious": false, "desired_opening": 7}}
JSON فقط:"""
        try:
            r = client.chat.completions.create(
                model=EVAL_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=150,
            )
            raw = r.choices[0].message.content.strip().replace("```json","").replace("```","")
            return json.loads(raw)
        except:
            return {"topic": None, "has_stats": False, "has_religious": False, "desired_opening": 0}

    def retrieve(self, query, top_k=3, verbose=True):
        """يسترجع أقرب الأمثلة مع تطبيق نظام Bonus"""
        meta = self._detect_query_meta(query)
        if verbose:
            print(f"  📌 الموضوع: {meta.get('topic')} | إحصائيات: {meta.get('has_stats')} | نمط البداية: {meta.get('desired_opening')}")

        query_emb    = EMBED_MODEL.encode(["query: " + query], normalize_embeddings=True)
        similarities = cosine_similarity(query_emb, self.embeddings)[0]

        topic  = meta.get("topic")
        scores = []
        for i, row in self.df.iterrows():
            score = similarities[i]
            if topic and row.get("topic") == topic:
                score += 0.15   # bonus الموضوع
            if meta.get("has_stats") and row.get("has_stats"):
                score += 0.05   # bonus الإحصائيات
            if meta.get("has_religious") and row.get("has_religious"):
                score += 0.05   # bonus العبارات الدينية
            scores.append((score, similarities[i], i))

        scores.sort(reverse=True)
        results = []
        for final_score, raw_sim, idx in scores[:top_k]:
            row  = self.df.iloc[idx]
            mark = "✓" if row.get("topic") == topic else " "
            if verbose:
                print(f"  {mark} {str(row.get('title_clean',''))[:45]} | دلالي:{raw_sim:.2f} نهائي:{final_score:.2f}")
            results.append({
                "title"          : row.get("title_clean", ""),
                "text"           : row.get("clean_text", ""),
                "topic"          : row.get("topic", ""),
                "opening_pattern": row.get("opening_pattern", 0),
                "has_stats"      : row.get("has_stats", False),
                "score"          : final_score,
                "similarity"     : raw_sim,
            })
        return results, meta

    def format_examples(self, examples, chars=600):
        out = ""
        for i, ex in enumerate(examples, 1):
            out += f"\n--- مثال {i} (الموضوع: {ex['topic']} | التشابه: {ex['similarity']:.2f}) ---\n"
            out += str(ex["text"])[:chars] + "...\n"
        return out

print("✅ كلاس SmartRAG جاهز")

In [ ]:
# بناء الـ RAG — شغّل مرة واحدة
rag = SmartRAG(df)
print("✅ RAG جاهز للاستخدام")

---
## ٤. البصمة الأسلوبية الكاملة

In [ ]:
STYLE_FINGERPRINT = """
## بصمة أسلوب أمين منطقة الرياض — مستخرجة من تحليل الداتا الحقيقية

### أنماط بداية النص (اختر الأنسب للموضوع):
1. الاستمرارية  : "امتدادًا لـ..." / "في رحلة تحول..."
2. التعليل      : "لأن... ولأن... ولذا..."
3. السياق       : "في رحلة تحول المملكة نحو رؤية 2030..."
4. المفارقة     : "كثير من المدن... ولكن فقط المدن المتميزة..."
5. الحدث المباشر: "سعدنا هذا الأسبوع بـ..." / "أعلنت أمانة الرياض..."
6. السؤال       : "ما الذي يجعل...؟" / "إذا سألت أي...؟"
7. الإحصاء      : "في مدينة تمتد على 380 ألف كيلومتر..." / "أكثر من X..."

### روابط الجمل الأكثر استخداماً (وزّعها بالتناوب):
حيث (57x) — كما (35x) — بل (35x) — ومن (31x) — وقد (23x) — ولكن (21x)
← لا تكرر نفس الرابط في جملتين متتاليتين

### مفردات لازمة يجب أن تظهر:
"جودة الحياة" — "منظومة" — "تكامل" — "نقلة نوعية" — "شركاء" — "مرحلة تاريخية"
"الارتقاء" — "المنظور الحضاري" — "رؤية 2030" — "التحول" — "المستدامة" — "الريادة"

### العبارات الدينية والوطنية (توزيعها الصحيح):
- في البداية كسياق: "بدعم مولاي خادم الحرمين الشريفين الملك سلمان بن عبدالعزيز -حفظه الله-،
  وتوجيهات سمو سيدي ولي العهد الأمير محمد بن سلمان -حفظه الله-"
- داخل النص عند إنجاز  : "ولله الحمد"
- عند مستهدف مستقبلي   : "بإذن الله"
- في النهاية            : "وفقًا لتوجيهات القيادة الرشيدة -حفظها الله-"

### أنماط الخاتمة الصحيحة (اختر واحدة وطوّعها):
- "ونأمل بحول الله أن يكون... سببًا في..."
- "ونسأل الله تعالى أن يوفقنا لما فيه الخير لمدينتنا..."
- "وسنواصل بإذن الله... وفقًا لتوجيهات القيادة الرشيدة حفظها الله"
- "ولله الحمد على ما تحقق، وسنواصل استكمال محطات هذه الرحلة"
- "لأن إسعادهم ورضاهم هو الوفاء بأسمى صوره"

### قواعد لا تُكسر أبداً:
1. لا كلمة لاتينية أو غير عربية في أي مكان
2. "حيث" مرة واحدة فقط لكل فقرة
3. "من خلال" مرة واحدة فقط في النص كله
4. الجملة الأولى لا تتجاوز 20 كلمة
5. ممنوع تماماً: وفي النهاية / وأخيراً / وفي الختام / وبالتالي / وبناءً على ذلك
6. ممنوع في المقالات: "بسم الله الرحمن الرحيم" (للكلمات الرسمية فقط)
7. الأسماء الحرفية الصحيحة فقط — لا تخترع أسماء أو ألقاباً
8. الأرقام بالأرقام لا بالحروف (380 ألف — 7 ملايين)
9. لا تكرر نفس الفكرة بصياغات مختلفة — تقدّم للفكرة التالية
10. كل فقرة تنتهي بفكرة مكتملة
"""

print("✅ البصمة الأسلوبية محملة")
print(f"📏 حجمها: {len(STYLE_FINGERPRINT)} حرف")

---
## ٥. الدوال الرئيسية

In [ ]:
def extract_ideas(text):
    """الخطوة ١: يستخرج الأفكار الرئيسية بدقة (temperature منخفضة)"""
    r = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "أنت مساعد يستخرج النقاط الرئيسية من النصوص. "
                    "استخرج الأفكار الجوهرية فقط كنقاط مرقمة بالعربية. "
                    "احتفظ بجميع الأرقام والإحصاءات. لا تضف أي شيء إضافي."
                )
            },
            {"role": "user", "content": f"استخرج الأفكار الرئيسية:\n\n{text}"}
        ],
        temperature=0.1,
        max_tokens=600,
    )
    return r.choices[0].message.content.strip()


def rewrite_in_style(ideas, examples_text, meta, text_type="مقال"):
    """الخطوة ٢: يعيد الكتابة من الصفر بالأسلوب الكامل (temperature متوسطة)"""
    pattern      = meta.get("desired_opening", 0)
    opening_hint = f"ابدأ بنمط البداية رقم {pattern}." if pattern else ""
    stats_hint   = "النص يحتاج إحصائيات — ادمجها بشكل طبيعي ضمن السياق." if meta.get("has_stats") else ""

    opening_rule = (
        "ابدأ بـ 'بسم الله الرحمن الرحيم' ثم 'أصحاب المعالي، الحضور الكريم'"
        if text_type == "كلمة رسمية"
        else f"ابدأ مباشرة بالموضوع. {opening_hint} لا تبدأ بـ 'بسم الله' في المقالات."
    )

    prompt = f"""أنت متخصص في محاكاة أسلوب كتابة أمين منطقة الرياض بدقة تامة.

{STYLE_FINGERPRINT}

## نصوص حقيقية من الداتا — مختارة لأنها الأقرب لموضوع نصك:
{examples_text}

## تعليمات الكتابة:
- {opening_rule}
- {stats_hint}
- اكتب النص من الصفر بناءً على الأفكار المُعطاة، لا تعدّل الجمل الأصلية
- وزّع العبارات الدينية والوطنية بشكل طبيعي غير مقحم
- اذكر القيادة في البداية كسياق داعم فقط
- اختر خاتمة من أنماط الخاتمة وطوّعها للموضوع
- لا تكرر أي فعل أو وصف أكثر من مرتين في النص كله

أجب بالنص فقط بدون أي تعليق أو مقدمة."""

    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": f"""اكتب {text_type} بأسلوب أمين منطقة الرياض بناءً على هذه الأفكار:

{ideas}

تذكير بالقواعد الصارمة:
- لا حرف لاتيني أبداً
- "حيث" مرة واحدة لكل فقرة
- "من خلال" مرة واحدة في النص كله
- ممنوع: وفي النهاية / وأخيراً / وبالتالي / وبناءً على ذلك
- الخاتمة من الأنماط المحددة في التعليمات"""
            }
        ],
        temperature=0.6,
        max_tokens=2500,
    )
    return r.choices[0].message.content.strip()


def review_and_fix(text):
    """الخطوة ٣: مراجعة تلقائية وتصحيح الأخطاء الأسلوبية"""
    issues = []

    # فحص تكرار "من خلال"
    count = text.count("من خلال")
    if count > 1:
        issues.append(f'"من خلال" تكررت {count} مرات — أبقِ الأولى وغيّر الباقي لـ: عبر / بواسطة / في إطار')

    # فحص الكلمات اللاتينية
    latin = re.findall(r'[a-zA-Z]+', text)
    if latin:
        issues.append(f'كلمات لاتينية: {latin} — استبدلها بالعربي')

    # فحص طول الجملة الأولى
    first = text.split("،")[0].split(".")[0]
    if len(first.split()) > 20:
        issues.append("الجملة الأولى أكثر من 20 كلمة — اختصرها")

    # فحص الكلمات الممنوعة
    banned = ["وفي النهاية", "وأخيراً", "وفي الختام", "وبالتالي", "وبناءً على ذلك"]
    found_banned = [w for w in banned if w in text]
    if found_banned:
        issues.append(f'كلمات ممنوعة: {found_banned} — احذفها أو أعد الصياغة')

    if not issues:
        return text, []

    # إرسال للتصحيح التلقائي
    prompt = f"""صحّح النص بناءً على هذه الملاحظات فقط — لا تغيّر شيئاً آخر:
{chr(10).join(issues)}

النص:
{text}

النص المصحح:"""
    r = client.chat.completions.create(
        model=EVAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0, max_tokens=2000,
    )
    return r.choices[0].message.content.strip(), issues


def transform_text(input_text, text_type="مقال", verbose=True):
    """
    الدالة الرئيسية — تحوّل النص بثلاث خطوات:
      ١. RAG ذكي لجلب أقرب الأمثلة + استخراج الأفكار
      ٢. إعادة الكتابة الكاملة بالأسلوب
      ٣. مراجعة تلقائية وتصحيح

    المعاملات:
        input_text : النص المراد تحويله
        text_type  : مقال / كلمة رسمية / تغريدة / فقرة / تقرير
        verbose    : True لعرض تفاصيل كل خطوة
    """
    if not input_text.strip():
        return "⚠️ الرجاء إدخال نص", {}, []

    if verbose: print("🔍 الخطوة ١: تحليل وجلب الأمثلة من الداتا...")
    examples, meta = rag.retrieve(input_text, verbose=verbose)
    formatted       = rag.format_examples(examples)

    if verbose: print("\n✍️  الخطوة ٢أ: استخراج الأفكار الرئيسية...")
    ideas = extract_ideas(input_text)
    if verbose:
        print("📌 الأفكار المستخرجة:")
        print(ideas)

    if verbose: print("\n🖊️  الخطوة ٢ب: إعادة الكتابة بالأسلوب...")
    draft = rewrite_in_style(ideas, formatted, meta, text_type)

    if verbose: print("\n🔎 الخطوة ٣: مراجعة وتصحيح تلقائي...")
    result, fixes = review_and_fix(draft)
    if verbose:
        if fixes:
            for f in fixes: print(f"  ⚠️  {f}")
        else:
            print("  ✅ النص نظيف — لا إصلاحات مطلوبة")

    return result, meta, examples


def display_comparison(original, converted):
    """يعرض مقارنة واضحة بين النص الأصلي والمحوّل"""
    sep = '=' * 60
    print(f'\n{sep}\n📄 النص الأصلي:\n{sep}')
    print(original)
    print(f'\n{sep}\n✨ النص بأسلوب الأمير:\n{sep}')
    print(converted)
    print(f'{sep}\n')


def save_result(original, converted, filename="output.txt"):
    """يحفظ النتيجة في ملف نصي"""
    sep = '=' * 60
    content = (
        f'{sep}\nالنص الأصلي:\n{sep}\n{original}\n\n'
        f'{sep}\nالنص بأسلوب الأمير:\n{sep}\n{converted}\n'
    )
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"✅ تم الحفظ في: {filename}")


print("✅ جميع الدوال جاهزة")

---
## ٦. الواجهة التفاعلية

In [ ]:
# ── الواجهة التفاعلية الكاملة ────────────────────────────────────────────────

title_lbl = widgets.HTML(
    value='<h2 style="color:#1a5276;font-family:Arial;direction:rtl">🏙️ محاكي أسلوب أمين منطقة الرياض</h2>'
)

input_box = widgets.Textarea(
    placeholder="أدخل النص المراد تحويله هنا...",
    layout=widgets.Layout(width="100%", height="160px"),
)

type_selector = widgets.Dropdown(
    options=["مقال", "كلمة رسمية", "تغريدة", "فقرة قصيرة", "تقرير"],
    value="مقال",
    description="نوع النص:",
    layout=widgets.Layout(width="240px"),
)

verbose_toggle = widgets.Checkbox(
    value=False,
    description="عرض التفاصيل",
    layout=widgets.Layout(width="160px"),
)

run_btn = widgets.Button(
    description="🖊️ حوّل النص",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="36px"),
)

save_btn = widgets.Button(
    description="💾 احفظ النتيجة",
    button_style="success",
    layout=widgets.Layout(width="160px", height="36px"),
    disabled=True,
)

status_out = widgets.Output()
debug_out  = widgets.Output()
result_out = widgets.Output()

_last_result = {"text": "", "original": ""}


def on_run(b):
    text = input_box.value.strip()
    if not text:
        with status_out:
            clear_output()
            print("⚠️ أدخل نصاً أولاً")
        return

    run_btn.disabled  = True
    save_btn.disabled = True
    with status_out:
        clear_output()
        print("⏳ جاري المعالجة (٣ خطوات)...")
    result_out.clear_output()
    debug_out.clear_output()

    verbose = verbose_toggle.value

    with debug_out:
        result, meta, examples = transform_text(
            text, type_selector.value, verbose=verbose
        )

    _last_result["text"]     = result
    _last_result["original"] = text

    with status_out:
        clear_output()
        print(f"✅ اكتمل | الموضوع: {meta.get('topic')} | الأمثلة المستخدمة: {len(examples)}")

    with result_out:
        clear_output()
        display(Markdown("### 📄 النص النهائي"))
        display(Markdown(
            f"<div dir='rtl' style='line-height:2;font-size:15px;"
            f"background:#f8f9fa;padding:20px;border-radius:8px;"
            f"border-right:4px solid #1F4E79'>{result}</div>"
        ))
        display(Markdown("---"))
        display(Markdown("**الأمثلة المستخدمة من الداتا:**"))
        for ex in examples:
            display(Markdown(
                f"- **{ex['title']}** "
                f"(الموضوع: {ex['topic']} | التشابه: {ex['similarity']:.2f} | النقاط: {ex['score']:.2f})"
            ))

    run_btn.disabled  = False
    save_btn.disabled = False


def on_save(b):
    if _last_result["text"]:
        save_result(_last_result["original"], _last_result["text"])
        with status_out:
            clear_output()
            print("✅ تم الحفظ في output.txt")


run_btn.on_click(on_run)
save_btn.on_click(on_save)

display(widgets.VBox([
    title_lbl,
    widgets.HTML("<hr>"),
    widgets.HTML("<b style='direction:rtl'>📝 النص الأصلي:</b>"),
    input_box,
    widgets.HBox(
        [type_selector, verbose_toggle, run_btn, save_btn],
        layout=widgets.Layout(gap="12px", align_items="center", margin="8px 0"),
    ),
    status_out,
    debug_out,
    widgets.HTML("<hr>"),
    result_out,
], layout=widgets.Layout(padding="20px", width="100%")))

---
## 🧪 اختبارات جاهزة
### اختبار ١ — مقال (تحويل مباشر بدون واجهة)

In [ ]:
test_article = """
تم إطلاق مشروع جديد لتطوير الخدمات الرقمية في المدينة بهدف تسهيل الوصول للخدمات الحكومية
وتحسين تجربة المستخدم. يشمل المشروع توحيد المنصات الرقمية وربط البيانات بين الجهات المختلفة،
بالإضافة إلى تطوير واجهات استخدام حديثة تعتمد على الذكاء الاصطناعي.
ويستهدف المشروع رفع كفاءة الأداء الحكومي وتقليل الوقت المستغرق في إنجاز المعاملات
بما يسهم في تحسين جودة الحياة وتعزيز رضا المستفيدين.
"""

result, meta, examples = transform_text(test_article, "مقال", verbose=True)
display_comparison(test_article.strip(), result)

### اختبار ٢ — كلمة رسمية

In [ ]:
test_speech = """
أرحب بكم في هذا المؤتمر الخاص بتطوير المدن.
هذا المؤتمر يجمع خبراء من مختلف أنحاء العالم.
موضوعاته تتناول أحدث التوجهات في إدارة المدن الذكية.
نتطلع لتبادل الخبرات والتعاون المثمر بين المشاركين.
"""

result, meta, examples = transform_text(test_speech, "كلمة رسمية", verbose=False)
display_comparison(test_speech.strip(), result)

---
## ٧. إضافة بيانات جديدة

> أضف نصوصاً جديدة من كتابات الأمين — تُضاف للـ CSV وتُعاد بناء الـ Embeddings تلقائياً

In [ ]:
def classify_new_text(title, text):
    """يصنّف النص الجديد تلقائياً بالـ AI"""
    prompt = f"""حلّل هذا النص وأرجع JSON فقط:

العنوان: {title}
النص: {text[:500]}

{{
  "topic": "اختر من: البنية التحتية، جودة الحياة، التكريم والجوائز، الزيارات والفعاليات، رؤية 2030، البيئة والاستدامة، التقنية والابتكار، الإدارة البلدية، المجتمع والثقافة، الإنجازات والأرقام",
  "opening_pattern": "رقم 1-7",
  "has_stats": "true/false",
  "has_religious": "true/false",
  "has_national": "true/false"
}}
JSON فقط."""
    try:
        r = client.chat.completions.create(
            model=EVAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=200,
        )
        raw = r.choices[0].message.content.strip().replace("```json","").replace("```","")
        return json.loads(raw)
    except:
        return {"topic":"غير محدد","opening_pattern":0,"has_stats":False,"has_religious":False,"has_national":False}


def add_text(title, text, text_type="مقال"):
    """يضيف نصاً جديداً للـ CSV ويُعيد بناء الـ Embeddings"""
    global df, rag

    print("⏳ تصنيف النص...")
    meta = classify_new_text(title, text)

    new_id  = int(df["id"].max()) + 1 if "id" in df.columns else len(df) + 1
    new_row = {
        "id"             : new_id,
        "title_clean"    : title,
        "clean_text"     : text,
        "type_ar"        : text_type,
        "text_len"       : len(text),
        "topic"          : meta["topic"],
        "opening_pattern": meta["opening_pattern"],
        "has_stats"      : meta["has_stats"],
        "has_religious"  : meta["has_religious"],
        "has_national"   : meta.get("has_national", False),
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

    print("⏳ تحديث الـ Embeddings...")
    rag = SmartRAG(df)

    print(f"✅ تمت الإضافة | ID: {new_id} | الموضوع: {meta['topic']}")
    return new_row


print("✅ دوال إضافة البيانات جاهزة")

In [ ]:
# ── أضف نصك هنا ─────────────────────────────────────────────────────────────
العنوان_الجديد = "عنوان النص هنا"

النص_الجديد = """
اكتب النص الكامل هنا...
"""

النوع = "مقال"   # مقال / كلمة رسمية / تغريدة

# ── شغّل الإضافة ─────────────────────────────────────────────────────────────
# result = add_text(العنوان_الجديد, النص_الجديد, النوع)
# print(f"\nتفاصيل الإضافة:")
# for k, v in result.items():
#     print(f"  {k}: {v}")

In [ ]:
# حالة الداتا الحالية
print(f"إجمالي النصوص: {len(df)}")
if 'topic' in df.columns:
    print("\nتوزيع المواضيع:")
    print(df['topic'].value_counts().to_string())

---
## 📌 ملاحظات مهمة

| الموضوع | التفصيل |
|---------|---------|
| **الداتا** | الهدف 200+ نص للحصول على نتائج مثالية |
| **RAG** | Embeddings دلالية عميقة (multilingual-e5-base) |
| **Bonus** | نفس الموضوع +0.15 \| نفس الخصائص +0.05 |
| **المراجعة** | تصلح "من خلال" والكلمات اللاتينية والكلمات الممنوعة تلقائياً |
| **النموذج** | `llama-3.3-70b` للكتابة \| `llama-3.1-8b` للتحليل والمراجعة |
| **نوع النص** | يؤثر على قاعدة البداية (بسم الله للكلمات الرسمية فقط) |
| **الحفظ** | زر "احفظ النتيجة" يصدر ملف output.txt |